# Deck, side, and home planet

A cabin string `deck/number/side` gives three fields. The deck and the home planet are almost the same fact.

|  | A | B | C | D | E | F | G | T | blank cabin |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Earth | 0 | 0 | 0 | 0 | 395 | 1614 | 2498 | 0 | 95 |
| Europa | 252 | 766 | 734 | 186 | 128 | 0 | 0 | 4 | 61 |
| Mars | 0 | 0 | 0 | 282 | 330 | 1110 | 0 | 0 | 37 |
| blank planet | 4 | 13 | 13 | 10 | 23 | 70 | 61 | 1 | 6 |

Earth appears on E, F, and G. Europa appears on A, B, C, D, E, and four times on T. Mars appears on D, E, and F. Deck B is 766 Europa passengers plus 13 passengers whose planet is blank: $766 + 13 = 779$. Deck T has five passengers.

The side is a separate cut. Port and starboard both have thousands of passengers.

$$
\begin{aligned}
\text{port } P &: 1898 \text{ transported of } 4206 \\
\text{starboard } S &: 2380 \text{ transported of } 4288 \\
\text{blank cabin} &: 100 \text{ transported of } 199
\end{aligned}
$$

Home planet, on its own:

$$
\begin{aligned}
\text{Earth} &: 1951 \text{ of } 4602 \\
\text{Europa} &: 1404 \text{ of } 2131 \\
\text{Mars} &: 920 \text{ of } 1759 \\
\text{blank} &: 103 \text{ of } 201
\end{aligned}
$$

Europa is the high planet. Deck B and deck C are the high decks, and they are Europa decks. Maham is the reminder printed on row 1: Europa, deck B, side P, and she stayed.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Rebuild the planet-by-deck table and the side counts.
from collections import Counter
table = Counter()
sides = Counter()
side_transported = Counter()
for row in train:
    planet = row["HomePlanet"]
    if row["Cabin"] == "":
        deck, side = "", ""
    else:
        deck, _, side = row["Cabin"].split("/")
    table[(planet, deck)] += 1
    sides[side] += 1
    side_transported[side] += row["Transported"] == "True"
planets = ["Earth", "Europa", "Mars", ""]
decks = ["A", "B", "C", "D", "E", "F", "G", "T", ""]
for planet in planets:
    counts = [table[(planet, deck)] for deck in decks]
    print((planet or "blank"), counts, sum(counts))
assert table[("Earth", "G")] == 2498
assert table[("Earth", "F")] == 1614
assert table[("Earth", "B")] == 0
assert table[("Europa", "B")] == 766
assert table[("Europa", "C")] == 734
assert table[("Europa", "F")] == 0
assert table[("Mars", "F")] == 1110
assert table[("Mars", "B")] == 0
assert table[("", "B")] == 13
assert 766 + 13 == 779
assert sides["P"] == 4206 and side_transported["P"] == 1898
assert sides["S"] == 4288 and side_transported["S"] == 2380
assert sides[""] == 199 and side_transported[""] == 100
planets_only = Counter(row["HomePlanet"] for row in train)
moved = Counter(
    row["HomePlanet"] for row in train if row["Transported"] == "True"
)
print("Earth", moved["Earth"], planets_only["Earth"])
print("Europa", moved["Europa"], planets_only["Europa"])
print("Mars", moved["Mars"], planets_only["Mars"])
assert (moved["Earth"], planets_only["Earth"]) == (1951, 4602)
assert (moved["Europa"], planets_only["Europa"]) == (1404, 2131)
assert (moved["Mars"], planets_only["Mars"]) == (920, 1759)
assert (moved[""], planets_only[""]) == (103, 201)


Earth [0, 0, 0, 0, 395, 1614, 2498, 0, 95] 4602
Europa [252, 766, 734, 186, 128, 0, 0, 4, 61] 2131
Mars [0, 0, 0, 282, 330, 1110, 0, 0, 37] 1759
blank [4, 13, 13, 10, 23, 70, 61, 1, 6] 201
Earth 1951 4602
Europa 1404 2131
Mars 920 1759


## Pitfall

A chart of deck alone makes deck B look like a cause. Of the 779 people on deck B, 766 are from Europa and 13 have a blank planet. Maham Ofracculy is one Europa passenger on deck B, and `Transported` on her row is `False`.

## Summary

Read deck and planet together. Earth is G, F, and E. Europa is A through E. Mars is D, E, and F. Starboard is higher than port: 2380 of 4288 against 1898 of 4206.
